# Conjugate Media — Model A (Trending) + Model B (Audience)

End-to-end training notebook for the two models you defined.

| Model | Goal | Data file |
|-------|------|-----------|
| **A** | Predict what is trending / engagement from **title, description, category, tags** + content meta | `ml_ready/model_A_trending.csv` |
| **B** | Infer **audience behaviour** (platform-specific) from **user posts only** (no trending) | `ml_ready/model_B_audience_posts.csv` |

### Engineered targets (already in the CSVs)
- `engagement_score` — composite 0–100 (main target)
- `breakout_log_ratio` = ln((views+1)/(subs+1))
- `quality_reaction_rate` = (likes+comments+all_reactions)/views
- `virality_index` = views + 15×likes + 50×comments + 100×shares
- `content_type`, `duration_sec`, `is_vertical_reel`, `is_short_form`, `post_hour`, …

**You do not need to re-scrape.** Everything uses the processed files.


## 0. Imports & paths

In [2]:
import warnings
warnings.filterwarnings('ignore')

import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier, GradientBoostingRegressor
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    accuracy_score, f1_score, classification_report, confusion_matrix
)
from sklearn.cluster import KMeans
from sklearn.decomposition import TruncatedSVD

try:
    import joblib
except ImportError:
    joblib = None

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (11, 5)

DATA_DIR = 'C:/Users/saura/OneDrive/Desktop/projects/sem7/EngageIQfinal/EngageIQ/conjugate-media/server/data/ml_ready'
MODEL_DIR = 'C:/Users/saura/OneDrive/Desktop/projects/sem7/EngageIQfinal/EngageIQ/conjugate-media/server/ml_models'
import os
os.makedirs(MODEL_DIR, exist_ok=True)

print('Ready')


ModuleNotFoundError: No module named 'pandas'

## 1. Load ML-ready datasets

In [ ]:
df_A = pd.read_csv(f'{DATA_DIR}/model_A_trending.csv')
df_B = pd.read_csv(f'{DATA_DIR}/model_B_audience_posts.csv')

print('Model A (trending):', df_A.shape)
print(df_A['platform'].value_counts())
print()
print('Model B (audience posts):', df_B.shape)
print(df_B['platform'].value_counts())
print()
print('Model A columns:', list(df_A.columns))


In [ ]:
# Quick look at engineered metrics
print('=== Model A engagement_score ===')
print(df_A['engagement_score'].describe().round(2))
print()
print('=== Model B engagement_score ===')
print(df_B['engagement_score'].describe().round(2))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
df_A['engagement_score'].dropna().hist(bins=20, ax=axes[0], color='steelblue', edgecolor='white')
axes[0].set_title('Model A — engagement_score (trending)')
df_B['engagement_score'].dropna().hist(bins=25, ax=axes[1], color='seagreen', edgecolor='white')
axes[1].set_title('Model B — engagement_score (audience posts)')
plt.tight_layout()
plt.show()


## 2. Shared helpers (text clean, metrics, train/eval)

In [ ]:
def clean_text(s):
    if pd.isna(s):
        return ''
    s = str(s).lower()
    s = re.sub(r'http\S+|www\S+', ' ', s)
    s = re.sub(r'[^a-z0-9\s#@]', ' ', s)
    s = re.sub(r'\s+', ' ', s).strip()
    return s

def combine_text(row, cols=('title', 'description', 'category', 'tags')):
    parts = [clean_text(row.get(c, '')) for c in cols]
    return ' '.join(p for p in parts if p)

def regression_report(name, y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = mean_squared_error(y_true, y_pred) ** 0.5
    r2 = r2_score(y_true, y_pred)
    print(f'{name}')
    print(f'  MAE  = {mae:.3f}')
    print(f'  RMSE = {rmse:.3f}')
    print(f'  R²   = {r2:.3f}')
    return {'name': name, 'MAE': mae, 'RMSE': rmse, 'R2': r2}

def make_high_label(series, quantile=0.60):
    """Binary: 1 = high engagement (above quantile), 0 = low/mid."""
    thr = series.quantile(quantile)
    return (series >= thr).astype(int), thr


---
# MODEL A — What is trending?

**Idea:** Use title + description + category + tags (text) and content meta  
(duration, short-form, reel, posting time, platform) to predict **engagement_score**.

We train:
1. **Regression** → predict continuous `engagement_score`
2. **Classification** → predict High vs Low engagement


### A1. Prepare features

In [ ]:
# Keep rows with a valid target
A = df_A.dropna(subset=['engagement_score']).copy()
print('Rows with engagement_score:', len(A))

# Combined text for TF-IDF
A['text'] = A.apply(combine_text, axis=1)
A['text_len'] = A['text'].str.len()

# Fill meta
for c in ['duration_sec', 'post_hour', 'post_dayofweek', 'is_weekend',
          'is_vertical_reel', 'is_short_form', 'tags_count']:
    if c in A.columns:
        A[c] = A[c].fillna(0)

A['platform'] = A['platform'].fillna('unknown')
A['content_type'] = A['content_type'].fillna('unknown')

print('Sample text:', A['text'].iloc[0][:120], '...')
print('content_type:', A['content_type'].value_counts().to_dict())


In [ ]:
# Feature groups
text_col = 'text'
cat_cols_A = ['platform', 'content_type']
num_cols_A = [
    'duration_sec', 'tags_count', 'text_len',
    'post_hour', 'post_dayofweek', 'is_weekend',
    'is_vertical_reel', 'is_short_form'
]
num_cols_A = [c for c in num_cols_A if c in A.columns]

y_A = A['engagement_score']
X_A = A[cat_cols_A + num_cols_A + [text_col]]

X_train_A, X_test_A, y_train_A, y_test_A = train_test_split(
    X_A, y_A, test_size=0.25, random_state=42
)
print('Train:', len(y_train_A), '| Test:', len(y_test_A))


### A2. Build pipelines (TF-IDF text + meta → model)

In [ ]:
def make_model_A_pipeline(regressor):
    """Text (TF-IDF) + categorical + numeric → regressor."""
    preprocess = ColumnTransformer(
        transformers=[
            ('text', TfidfVectorizer(max_features=1500, ngram_range=(1, 2), min_df=1), text_col),
            ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols_A),
            ('num', Pipeline([
                ('impute', SimpleImputer(strategy='median')),
                ('scale', StandardScaler()),
            ]), num_cols_A),
        ],
        remainder='drop'
    )
    return Pipeline([
        ('prep', preprocess),
        ('model', regressor)
    ])

pipe_A_ridge = make_model_A_pipeline(Ridge(alpha=1.0))
pipe_A_rf    = make_model_A_pipeline(RandomForestRegressor(
    n_estimators=120, max_depth=10, random_state=42, n_jobs=-1
))
pipe_A_gb    = make_model_A_pipeline(GradientBoostingRegressor(
    n_estimators=100, max_depth=4, learning_rate=0.08, random_state=42
))

print('Model A pipelines ready')


### A3. Train & evaluate regression

In [ ]:
pipe_A_ridge.fit(X_train_A, y_train_A)
pipe_A_rf.fit(X_train_A, y_train_A)
pipe_A_gb.fit(X_train_A, y_train_A)

pred_ridge = pipe_A_ridge.predict(X_test_A)
pred_rf    = pipe_A_rf.predict(X_test_A)
pred_gb    = pipe_A_gb.predict(X_test_A)

results_A = []
results_A.append(regression_report('A · Ridge (text+meta)', y_test_A, pred_ridge))
results_A.append(regression_report('A · Random Forest', y_test_A, pred_rf))
results_A.append(regression_report('A · Gradient Boosting', y_test_A, pred_gb))
pd.DataFrame(results_A)


In [ ]:
# Predicted vs actual
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, pred, title in [
    (axes[0], pred_ridge, 'Ridge'),
    (axes[1], pred_rf, 'Random Forest'),
    (axes[2], pred_gb, 'Gradient Boosting'),
]:
    ax.scatter(y_test_A, pred, alpha=0.6, s=35)
    lims = [y_test_A.min(), y_test_A.max()]
    ax.plot(lims, lims, 'r--', lw=1.5)
    ax.set_xlabel('Actual engagement_score')
    ax.set_ylabel('Predicted')
    ax.set_title(f'Model A — {title}')
plt.tight_layout()
plt.show()


### A4. Classification: High vs Low engagement

In [ ]:
y_bin_A, thr_A = make_high_label(y_A, quantile=0.60)
print(f'High-engagement threshold (60th pct): {thr_A:.2f}')
print(y_bin_A.value_counts().rename({0: 'low/mid', 1: 'high'}))

X_train_Ac, X_test_Ac, y_train_Ac, y_test_Ac = train_test_split(
    X_A, y_bin_A, test_size=0.25, random_state=42, stratify=y_bin_A
)

def make_model_A_clf(clf):
    preprocess = ColumnTransformer(
        transformers=[
            ('text', TfidfVectorizer(max_features=1500, ngram_range=(1, 2), min_df=1), text_col),
            ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols_A),
            ('num', Pipeline([
                ('impute', SimpleImputer(strategy='median')),
                ('scale', StandardScaler()),
            ]), num_cols_A),
        ]
    )
    return Pipeline([('prep', preprocess), ('model', clf)])

pipe_A_clf = make_model_A_clf(RandomForestClassifier(
    n_estimators=120, max_depth=10, random_state=42, n_jobs=-1, class_weight='balanced'
))
pipe_A_clf.fit(X_train_Ac, y_train_Ac)
pred_Ac = pipe_A_clf.predict(X_test_Ac)

print('Accuracy:', round(accuracy_score(y_test_Ac, pred_Ac), 3))
print('F1     :', round(f1_score(y_test_Ac, pred_Ac), 3))
print()
print(classification_report(y_test_Ac, pred_Ac, target_names=['low/mid', 'high']))


In [ ]:
# Confusion matrix
cm = confusion_matrix(y_test_Ac, pred_Ac)
fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['low/mid', 'high'], yticklabels=['low/mid', 'high'], ax=ax)
ax.set_xlabel('Predicted')
ax.set_ylabel('Actual')
ax.set_title('Model A — High vs Low engagement')
plt.tight_layout()
plt.show()


### A5. Try Model A on a new title/description

In [ ]:
def predict_trending_score(title, description='', category='', tags='',
                           platform='youtube', content_type='video',
                           duration_sec=180, is_short_form=0, is_vertical_reel=0,
                           post_hour=12, post_dayofweek=2, model=pipe_A_gb):
    text = clean_text(' '.join([title, description, category, tags]))
    row = pd.DataFrame([{
        'platform': platform,
        'content_type': content_type,
        'duration_sec': duration_sec,
        'tags_count': len(tags.split()) if tags else 0,
        'text_len': len(text),
        'post_hour': post_hour,
        'post_dayofweek': post_dayofweek,
        'is_weekend': 1 if post_dayofweek >= 5 else 0,
        'is_vertical_reel': is_vertical_reel,
        'is_short_form': is_short_form,
        'text': text,
    }])
    score = float(model.predict(row)[0])
    label = 'HIGH' if score >= thr_A else 'low/mid'
    print(f'Predicted engagement_score = {score:.1f}  →  {label} trending potential')
    return score

# Examples
predict_trending_score(
    title='Official Trailer | Superstar Action Movie 2026',
    description='Biggest action film of the year with mind-blowing stunts',
    category='entertainment', tags='trailer movie action',
    platform='youtube', content_type='video', duration_sec=150
)

predict_trending_score(
    title='5 morning habits that changed my life',
    description='Simple productivity tips for creators',
    category='lifestyle', tags='productivity habits',
    platform='instagram', content_type='reel', duration_sec=45,
    is_short_form=1, is_vertical_reel=1
)


---
# MODEL B — Audience type (platform-specific)

**Idea:** Using **only the creator’s own posts** (no trending data), learn:
1. How engaging each post is (`engagement_score`)
2. What **audience style** the account has (breakout vs loyal, video-heavy vs text, etc.)

We use the same engineered metrics: breakout_log_ratio, quality_reaction_rate, virality_index, content type, timing, duration.


### B1. Prepare audience-post features

In [ ]:
B = df_B.dropna(subset=['engagement_score']).copy()
print('Rows with engagement_score:', len(B))
print(B.groupby('platform')['engagement_score'].agg(['count', 'mean', 'median']).round(2))

B['text'] = B.apply(lambda r: combine_text(r, cols=('title', 'description', 'category')), axis=1)
B['text_len'] = B['text'].str.len()

for c in ['duration_sec', 'post_hour', 'post_dayofweek', 'is_weekend',
          'is_vertical_reel', 'is_short_form', 'tags_count', 'subscribers',
          'breakout_log_ratio', 'quality_reaction_rate', 'virality_index']:
    if c in B.columns:
        B[c] = pd.to_numeric(B[c], errors='coerce')

B['platform'] = B['platform'].fillna('unknown')
B['content_type'] = B['content_type'].fillna('unknown')

# log virality for stability
B['log_virality'] = np.log1p(B['virality_index'].fillna(0))


In [ ]:
cat_cols_B = ['platform', 'content_type']
num_cols_B = [
    'duration_sec', 'text_len', 'tags_count',
    'post_hour', 'post_dayofweek', 'is_weekend',
    'is_vertical_reel', 'is_short_form',
    'subscribers', 'breakout_log_ratio', 'log_virality'
]
# Note: we do NOT put likes/comments/shares directly into features when predicting
# engagement_score (to avoid leakage). breakout & log_virality are derived signals
# that still help describe audience response patterns.
num_cols_B = [c for c in num_cols_B if c in B.columns]

y_B = B['engagement_score']
X_B = B[cat_cols_B + num_cols_B]  # meta-focused (audience behaviour)

X_train_B, X_test_B, y_train_B, y_test_B = train_test_split(
    X_B, y_B, test_size=0.20, random_state=42
)
print('Train:', len(y_train_B), '| Test:', len(y_test_B))
print('Numeric features:', num_cols_B)


### B2. Train engagement regressor (audience posts)

In [ ]:
def make_model_B_pipeline(regressor):
    preprocess = ColumnTransformer(
        transformers=[
            ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols_B),
            ('num', Pipeline([
                ('impute', SimpleImputer(strategy='median')),
                ('scale', StandardScaler()),
            ]), num_cols_B),
        ]
    )
    return Pipeline([('prep', preprocess), ('model', regressor)])

pipe_B_rf = make_model_B_pipeline(RandomForestRegressor(
    n_estimators=150, max_depth=12, random_state=42, n_jobs=-1
))
pipe_B_gb = make_model_B_pipeline(GradientBoostingRegressor(
    n_estimators=120, max_depth=4, learning_rate=0.08, random_state=42
))

pipe_B_rf.fit(X_train_B, y_train_B)
pipe_B_gb.fit(X_train_B, y_train_B)

pred_B_rf = pipe_B_rf.predict(X_test_B)
pred_B_gb = pipe_B_gb.predict(X_test_B)

results_B = []
results_B.append(regression_report('B · Random Forest', y_test_B, pred_B_rf))
results_B.append(regression_report('B · Gradient Boosting', y_test_B, pred_B_gb))
pd.DataFrame(results_B)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, pred, title in [
    (axes[0], pred_B_rf, 'Random Forest'),
    (axes[1], pred_B_gb, 'Gradient Boosting'),
]:
    ax.scatter(y_test_B, pred, alpha=0.4, s=20, c='seagreen')
    lims = [y_test_B.min(), y_test_B.max()]
    ax.plot(lims, lims, 'r--', lw=1.5)
    ax.set_xlabel('Actual engagement_score')
    ax.set_ylabel('Predicted')
    ax.set_title(f'Model B — {title}')
plt.tight_layout()
plt.show()


### B3. Feature importance (what drives this audience’s engagement?)

In [ ]:
# Feature names after one-hot
pre_B = pipe_B_rf.named_steps['prep']
cat_names = list(pre_B.named_transformers_['cat'].get_feature_names_out(cat_cols_B))
feat_names_B = list(cat_names) + list(num_cols_B)
importances_B = pd.Series(
    pipe_B_rf.named_steps['model'].feature_importances_,
    index=feat_names_B
).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(9, 7))
importances_B.tail(18).plot(kind='barh', ax=ax, color='seagreen')
ax.set_title('Model B — Feature importance (audience engagement)')
ax.set_xlabel('Importance')
plt.tight_layout()
plt.show()
print(importances_B.sort_values(ascending=False).head(12).round(4))


### B4. Audience segmentation (clustering)

Cluster posts by engagement behaviour to describe **audience type**:
- High breakout (viral reach relative to subs)
- High quality reactions
- Short-form / reel preference
- etc.


In [ ]:
cluster_features = [
    'engagement_score', 'breakout_log_ratio', 'quality_reaction_rate',
    'log_virality', 'is_short_form', 'is_vertical_reel', 'duration_sec'
]
cluster_features = [c for c in cluster_features if c in B.columns]

Cb = B[cluster_features].copy()
Cb = Cb.fillna(Cb.median(numeric_only=True))

scaler = StandardScaler()
X_cl = scaler.fit_transform(Cb)

n_clusters = 4
kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
B['audience_cluster'] = kmeans.fit_predict(X_cl)

print('Cluster sizes:')
print(B['audience_cluster'].value_counts().sort_index())

# Profile each cluster
profile = B.groupby('audience_cluster')[cluster_features + ['platform']].agg({
    **{c: 'mean' for c in cluster_features},
    'platform': lambda s: s.value_counts().index[0]  # dominant platform
}).round(3)
profile['n'] = B.groupby('audience_cluster').size()
print('\nCluster profiles (mean values):')
display(profile)


In [ ]:
# Name clusters in plain language
def label_cluster(row):
    if row['breakout_log_ratio'] > profile['breakout_log_ratio'].median() and row['engagement_score'] > profile['engagement_score'].median():
        return 'Viral Breakout'
    if row['quality_reaction_rate'] > profile['quality_reaction_rate'].median():
        return 'Loyal High-Reaction'
    if row.get('is_short_form', 0) > 0.5:
        return 'Short-Form / Reel Audience'
    return 'Steady / Broad Reach'

# Map cluster id → readable name using profile
cluster_names = {}
for cid, row in profile.iterrows():
    cluster_names[cid] = label_cluster(row)
print('Cluster labels:', cluster_names)

B['audience_type'] = B['audience_cluster'].map(cluster_names)
print()
print(B['audience_type'].value_counts())


In [ ]:
# Platform × audience type
ct = pd.crosstab(B['platform'], B['audience_type'], normalize='index') * 100
fig, ax = plt.subplots(figsize=(10, 5))
ct.plot(kind='bar', stacked=True, ax=ax, colormap='Set2')
ax.set_ylabel('% of posts')
ax.set_title('Model B — Audience type mix by platform')
ax.legend(title='Audience type', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()


### B5. Per-platform mini models (optional but useful)

In [ ]:
platform_results = []
for plat, g in B.groupby('platform'):
    if len(g) < 30:
        print(f'Skip {plat}: only {len(g)} rows')
        continue
    Xp = g[cat_cols_B + num_cols_B]
    yp = g['engagement_score']
    # drop platform col variance (all same) — keep content_type
    cat_p = ['content_type']
    num_p = num_cols_B
    Xp = g[cat_p + num_p]
    Xtr, Xte, ytr, yte = train_test_split(Xp, yp, test_size=0.25, random_state=42)
    pipe = Pipeline([
        ('prep', ColumnTransformer([
            ('cat', OneHotEncoder(handle_unknown='ignore'), cat_p),
            ('num', Pipeline([
                ('impute', SimpleImputer(strategy='median')),
                ('scale', StandardScaler())
            ]), num_p),
        ])),
        ('model', RandomForestRegressor(n_estimators=80, max_depth=8, random_state=42, n_jobs=-1))
    ])
    pipe.fit(Xtr, ytr)
    pred = pipe.predict(Xte)
    r2 = r2_score(yte, pred)
    mae = mean_absolute_error(yte, pred)
    platform_results.append({'platform': plat, 'n': len(g), 'MAE': round(mae, 3), 'R2': round(r2, 3)})
    print(f'{plat:12s}  n={len(g):4d}  MAE={mae:.3f}  R²={r2:.3f}')

pd.DataFrame(platform_results)


### B6. Engagement prediction **based on audience type**

Now that each post has an `audience_type`, we train a second Model B that:

1. Uses **audience_type** as a feature (Viral Breakout / Loyal / Short-Form / Steady…)
2. Predicts **engagement_score** for a new post *given* that audience

This answers: *“If my audience is mostly X, what engagement should I expect for this content?”*


In [ ]:
# Make sure audience_type exists (from clustering above)
assert 'audience_type' in B.columns, 'Run the clustering cells first'
print('Audience type distribution:')
print(B['audience_type'].value_counts())
print()

# Features for audience-aware engagement model
cat_cols_B2 = ['platform', 'content_type', 'audience_type']
num_cols_B2 = [
    'duration_sec', 'text_len', 'tags_count',
    'post_hour', 'post_dayofweek', 'is_weekend',
    'is_vertical_reel', 'is_short_form',
    'subscribers', 'breakout_log_ratio', 'log_virality'
]
num_cols_B2 = [c for c in num_cols_B2 if c in B.columns]

# Fill any missing audience_type
B['audience_type'] = B['audience_type'].fillna('Steady / Broad Reach')

y_B2 = B['engagement_score']
X_B2 = B[cat_cols_B2 + num_cols_B2]

X_train_B2, X_test_B2, y_train_B2, y_test_B2 = train_test_split(
    X_B2, y_B2, test_size=0.20, random_state=42
)
print('Train:', len(y_train_B2), '| Test:', len(y_test_B2))
print('Features:', cat_cols_B2 + num_cols_B2)


In [ ]:
def make_audience_engagement_pipeline(regressor):
    preprocess = ColumnTransformer(
        transformers=[
            ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols_B2),
            ('num', Pipeline([
                ('impute', SimpleImputer(strategy='median')),
                ('scale', StandardScaler()),
            ]), num_cols_B2),
        ]
    )
    return Pipeline([('prep', preprocess), ('model', regressor)])

pipe_B_audience = make_audience_engagement_pipeline(
    RandomForestRegressor(n_estimators=150, max_depth=12, random_state=42, n_jobs=-1)
)
pipe_B_audience.fit(X_train_B2, y_train_B2)

pred_B2 = pipe_B_audience.predict(X_test_B2)
_ = regression_report('B · Audience-aware engagement (RF)', y_test_B2, pred_B2)

# Compare with earlier Model B (without audience_type)
print('(Compare with earlier Model B RF which had no audience_type feature)')


In [ ]:
# Feature importance — does audience_type matter?
pre = pipe_B_audience.named_steps['prep']
cat_names = list(pre.named_transformers_['cat'].get_feature_names_out(cat_cols_B2))
feat_names = list(cat_names) + list(num_cols_B2)
imp = pd.Series(
    pipe_B_audience.named_steps['model'].feature_importances_,
    index=feat_names
).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(9, 7))
imp.tail(20).plot(kind='barh', ax=ax, color='darkorange')
ax.set_title('Model B — Audience-aware engagement: feature importance')
ax.set_xlabel('Importance')
plt.tight_layout()
plt.show()

# Highlight audience_type features
aud_imp = imp[[c for c in imp.index if 'audience_type' in c]].sort_values(ascending=False)
print('Importance of audience_type categories:')
print(aud_imp.round(4))


### B7. Predict engagement for a new post **given audience type**

Change the values below to simulate: *“My LinkedIn audience is Steady / Broad Reach — what score do I get for a text post?”*


In [ ]:
def predict_engagement_for_audience(
    platform='youtube',
    content_type='video',
    audience_type='Steady / Broad Reach',
    duration_sec=180,
    text_len=100,
    tags_count=3,
    post_hour=12,
    post_dayofweek=2,
    is_weekend=0,
    is_vertical_reel=0,
    is_short_form=0,
    subscribers=1_000_000,
    views_guess=100_000,   # used only to build breakout / virality proxies
    likes_guess=5_000,
    comments_guess=200,
    shares_guess=50,
    model=pipe_B_audience,
):
    """Predict engagement_score conditioned on audience_type."""
    breakout = np.log((views_guess + 1) / (subscribers + 1))
    virality = views_guess + 15 * likes_guess + 50 * comments_guess + 100 * shares_guess
    row = pd.DataFrame([{
        'platform': platform,
        'content_type': content_type,
        'audience_type': audience_type,
        'duration_sec': duration_sec,
        'text_len': text_len,
        'tags_count': tags_count,
        'post_hour': post_hour,
        'post_dayofweek': post_dayofweek,
        'is_weekend': is_weekend,
        'is_vertical_reel': is_vertical_reel,
        'is_short_form': is_short_form,
        'subscribers': subscribers,
        'breakout_log_ratio': breakout,
        'log_virality': np.log1p(virality),
    }])
    # Keep only columns the model expects
    row = row[cat_cols_B2 + num_cols_B2]
    score = float(model.predict(row)[0])
    print(f'Platform       : {platform}')
    print(f'Content type   : {content_type}')
    print(f'Audience type  : {audience_type}')
    print(f'Predicted engagement_score = {score:.1f}')
    return score

# --- Examples ---
print('Example 1: YouTube + Viral Breakout audience + video')
predict_engagement_for_audience(
    platform='youtube', content_type='video',
    audience_type='Viral Breakout',
    duration_sec=300, subscribers=8_630_000,
    views_guess=2_000_000, likes_guess=80_000
)
print()
print('Example 2: LinkedIn + Steady audience + text post')
predict_engagement_for_audience(
    platform='linkedin', content_type='text',
    audience_type='Steady / Broad Reach',
    duration_sec=0, is_short_form=0, subscribers=485_000,
    views_guess=20_000, likes_guess=500, comments_guess=40
)
print()
print('Example 3: Instagram + Viral Breakout + reel')
predict_engagement_for_audience(
    platform='instagram', content_type='reel',
    audience_type='Viral Breakout',
    duration_sec=30, is_short_form=1, is_vertical_reel=1,
    subscribers=3_662_163, views_guess=500_000, likes_guess=40_000
)


### B8. Engagement by audience type (what each audience typically scores)


In [ ]:
# Average actual engagement per audience type × platform
pivot = B.pivot_table(
    values='engagement_score',
    index='audience_type',
    columns='platform',
    aggfunc='mean'
).round(1)

print('Mean engagement_score by audience type × platform:')
display(pivot)

fig, ax = plt.subplots(figsize=(10, 4))
B.boxplot(column='engagement_score', by='audience_type', ax=ax)
ax.set_title('Engagement score distribution by audience type')
ax.set_xlabel('Audience type')
plt.suptitle('')
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()


### B9. How to use Model B in practice

```text
Step 1  Cluster / label the creator's past posts → audience_type per platform
Step 2  For a NEW post idea, pick:
          - platform
          - content_type (video / reel / text / …)
          - the audience_type you believe that platform has
Step 3  Call predict_engagement_for_audience(...) → engagement_score
```

The saved model `model_B_audience_engagement.joblib` expects the same columns.


---
## 3. Save trained models

In [ ]:
if joblib is not None:
    joblib.dump(pipe_A_gb, f'{MODEL_DIR}/model_A_trending_gb.joblib')
    joblib.dump(pipe_A_clf, f'{MODEL_DIR}/model_A_trending_clf.joblib')
    joblib.dump(pipe_B_rf, f'{MODEL_DIR}/model_B_audience_rf.joblib')
    joblib.dump(pipe_B_audience, f'{MODEL_DIR}/model_B_audience_engagement.joblib')  # audience-aware
    joblib.dump({
        'kmeans': kmeans,
        'scaler': scaler,
        'cluster_features': cluster_features,
        'cluster_names': cluster_names,
        'high_threshold_A': float(thr_A),
    }, f'{MODEL_DIR}/model_B_audience_clusters.joblib')
    print('Saved:')
    for f in sorted(os.listdir(MODEL_DIR)):
        if f.startswith('model_'):
            print(' ', f)
else:
    print('joblib not available — models trained in memory only')


---
## 4. Summary

### Model A (Trending)
- **Inputs:** title, description, category, tags (TF-IDF) + platform, content_type, duration, short-form, reel, posting time  
- **Targets:** `engagement_score` (regression) and High/Low class  
- **Use:** Score a new post idea *before* publishing → “will this trend?”

### Model B (Audience)
- **Inputs:** platform, content_type, duration, timing, subscribers, breakout_log_ratio, log_virality, …  
- **Targets:** `engagement_score` + **audience_type** clusters (Viral Breakout, Loyal High-Reaction, Short-Form, Steady)  
- **Use:** Understand *what kind of audience* each platform has for this creator, and which formats they reward  

### Key engineered columns (already in CSV)
| Column | Formula |
|--------|---------|
| breakout_log_ratio | ln((views+1)/(subs+1)) |
| quality_reaction_rate | (likes+comments+all_reactions)/views |
| virality_index | views + 15×likes + 50×comments + 100×shares |
| engagement_score | composite 0–100 target |

### Files produced
- `ml_ready/model_A_trending.csv`
- `ml_ready/model_B_audience_posts.csv` (+ per-platform splits)
- `ml_models/model_A_trending_gb.joblib`
- `ml_models/model_B_audience_rf.joblib`
- `ml_models/model_B_audience_clusters.joblib`

You can re-run any cell, change example titles in **A5**, or add more features from the raw JSON later.


### Model B — Audience-aware engagement prediction (NEW)
- After clustering posts into audience types, a second model predicts **engagement_score** using `audience_type` as a feature
- Function: `predict_engagement_for_audience(platform, content_type, audience_type, ...)`
- Saved as: `model_B_audience_engagement.joblib`


In [ ]:
print('=== DONE ===')
print(f'Model A rows used: {len(A)}')
print(f'Model B rows used: {len(B)}')
print(f'Model A high-engagement threshold: {thr_A:.2f}')
print('Audience types:', B['audience_type'].value_counts().to_dict())
